In [ ]:
pip install torch torchvision ultralytics onnx onnxruntime onnxslim torch-pruning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 90.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 106.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [ultralytics]


In [ ]:
# -*- coding: utf-8 -*-
from collections import defaultdict
from dataclasses import dataclass
import os
import random
import shutil
import zipfile

import cv2
import numpy as np
import torch
import yaml
from ultralytics import YOLO


# ==========================================
# CENTRAL CONFIGURATION
# ==========================================
@dataclass
class Config:
    # File Paths & Directories
    zip_filename: str = "heavy_model.zip"
    raw_dataset_dir: str = "drowsy_dataset"
    augmented_dataset_dir: str = "augmented_dataset"
    yolo_dataset_dir: str = "yolo_augmented_dataset"

    # Training Parameters
    base_model_yaml: str = "yolov8n.yaml"
    custom_model_yaml: str = "custom_scaled_yolov8.yaml"

    # Scale width multiplier to control model capacity and size directly:
    # Standard YOLOv8n width_multiple is 0.25.
    # Reducing to 0.125 cuts channel capacity by half across all layers.
    width_multiple: float = 0.18
    depth_multiple: float = 0.33

    img_size: int = 640
    epochs: int = 200
    patience: int = 25
    train_project: str = "/content/runs/detect/heavy_model_retrained"
    train_name: str = "drowsy"
    train_split_ratio: float = 0.8
    seed: int = 42

    # Class Definitions
    class_names: tuple = ("awake", "sleepy")

    # Export Settings
    onnx_output_filename: str = "scaled_model_int8.onnx"
    int8_quantization: bool = True

    @property
    def data_yaml_path(self) -> str:
        return os.path.join(self.yolo_dataset_dir, "data.yaml")

    @property
    def best_weights_path(self) -> str:
        return os.path.join(
            self.train_project, self.train_name, "weights", "best.pt"
        )


# Instantiate central config
CONFIG = Config()


# ==========================================
# STEP 1: EXTRACT DATASET
# ==========================================
def extract_dataset(cfg: Config = CONFIG):
    """Extract model zip archive if present."""
    print("STEP 1: Extracting dataset zip")
    if os.path.exists(cfg.zip_filename):
        with zipfile.ZipFile(cfg.zip_filename, "r") as zip_ref:
            zip_ref.extractall(".")


# ==========================================
# STEP 2a: LOCATE DATASET
# ==========================================
def locate_dataset(cfg: Config = CONFIG):
    """Locate the raw dataset directory containing class folders."""
    print("\nSTEP 2a: Locating dataset")
    if os.path.exists(cfg.raw_dataset_dir):
        return cfg.raw_dataset_dir

    alt_path = os.path.join("heavy_model", cfg.raw_dataset_dir)
    if os.path.exists(alt_path):
        return alt_path

    for root, dirs, _ in os.walk("."):
        if all(cls in dirs for cls in cfg.class_names):
            return os.path.dirname(os.path.join(root, cfg.class_names[0]))

    print("Dataset path not found")
    exit(1)


# ==========================================
# STEP 2b: AUGMENTATION HELPERS & PROCESSING
# ==========================================
def augment_image(image_path, output_dir, class_name):
    """Helper function to apply data augmentations to a single image."""
    img = cv2.imread(image_path)
    if img is None:
        return 0
    h, w = img.shape[:2]
    orig_name = os.path.splitext(os.path.basename(image_path))[0]

    cv2.imwrite(f"{output_dir}/{class_name}_{orig_name}_orig.jpg", img)
    cv2.imwrite(f"{output_dir}/{class_name}_{orig_name}_flip.jpg", cv2.flip(img, 1))
    cv2.imwrite(
        f"{output_dir}/{class_name}_{orig_name}_bright.jpg",
        cv2.convertScaleAbs(img, alpha=1.2, beta=20),
    )
    cv2.imwrite(
        f"{output_dir}/{class_name}_{orig_name}_dark.jpg",
        cv2.convertScaleAbs(img, alpha=0.8, beta=-20),
    )
    cv2.imwrite(
        f"{output_dir}/{class_name}_{orig_name}_blur.jpg",
        cv2.GaussianBlur(img, (5, 5), 0),
    )
    cv2.imwrite(
        f"{output_dir}/{class_name}_{orig_name}_sharp.jpg",
        cv2.filter2D(img, -1, np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]])),
    )

    for angle in [-10, 10]:
        matrix = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
        cv2.imwrite(
            f"{output_dir}/{class_name}_{orig_name}_rot{angle}.jpg",
            cv2.warpAffine(img, matrix, (w, h)),
        )
    return 8


def process_augmentations(dataset_path, cfg: Config = CONFIG):
    """Apply image augmentations across all classes."""
    print("\nSTEP 2b: Processing image augmentations")
    os.makedirs(cfg.augmented_dataset_dir, exist_ok=True)
    for cls in cfg.class_names:
        img_dir = os.path.join(dataset_path, cls)
        if os.path.exists(img_dir):
            for img_file in os.listdir(img_dir):
                if img_file.lower().endswith((".jpg", ".png", ".jpeg")):
                    augment_image(
                        os.path.join(img_dir, img_file),
                        cfg.augmented_dataset_dir,
                        cls,
                    )
    return cfg.augmented_dataset_dir


# ==========================================
# STEP 3: CREATE YOLO DATASET & YAML (GROUPED SPLIT)
# ==========================================
def create_yolo_dataset(cfg: Config = CONFIG):
    """Format augmented images into YOLO directory structure,

    grouping all augmentations of the same base image together to prevent leakage.
    """
    print("\nSTEP 3: Creating YOLO dataset and YAML configuration")
    for split in ["train", "val"]:
        os.makedirs(
            os.path.join(cfg.yolo_dataset_dir, "images", split), exist_ok=True
        )
        os.makedirs(
            os.path.join(cfg.yolo_dataset_dir, "labels", split), exist_ok=True
        )

    # Group augmented images by their base original image name
    groups = defaultdict(list)
    suffixes = [
        "_orig",
        "_flip",
        "_bright",
        "_dark",
        "_blur",
        "_sharp",
        "_rot-10",
        "_rot10",
    ]

    for f in os.listdir(cfg.augmented_dataset_dir):
        if not f.lower().endswith((".jpg", ".png", ".jpeg")):
            continue

        # Determine class name and ID
        class_id = None
        cls_name = None
        for idx, c in enumerate(cfg.class_names):
            if f.startswith(c + "_"):
                cls_name = c
                class_id = idx
                break
        if cls_name is None:
            continue

        # Strip class prefix and extension to parse base image name
        remainder = f[len(cls_name) + 1 :]
        name_no_ext = os.path.splitext(remainder)[0]

        orig_img_name = name_no_ext
        for sfx in sorted(suffixes, key=len, reverse=True):
            if name_no_ext.endswith(sfx):
                orig_img_name = name_no_ext[: -len(sfx)]
                break

        group_key = (cls_name, orig_img_name)
        groups[group_key].append(
            (os.path.join(cfg.augmented_dataset_dir, f), class_id)
        )

    # Shuffle groups (not individual augmented images) to prevent data leakage
    group_items = list(groups.items())
    random.seed(cfg.seed)
    random.shuffle(group_items)

    split_idx = int(cfg.train_split_ratio * len(group_items))
    train_groups = group_items[:split_idx]
    val_groups = group_items[split_idx:]

    splits = {
        "train": [item for _, samples in train_groups for item in samples],
        "val": [item for _, samples in val_groups for item in samples],
    }

    for split_name, samples in splits.items():
        for img_path, class_id in samples:
            name = os.path.basename(img_path)
            shutil.copy2(
                img_path,
                os.path.join(cfg.yolo_dataset_dir, "images", split_name, name),
            )
            txt_name = os.path.splitext(name)[0] + ".txt"
            label_path = os.path.join(
                cfg.yolo_dataset_dir, "labels", split_name, txt_name
            )
            with open(label_path, "w") as f:
                f.write(f"{class_id} 0.5 0.5 1.0 1.0\n")

    yaml_data = {
        "path": os.path.abspath(cfg.yolo_dataset_dir),
        "train": "images/train",
        "val": "images/val",
        "nc": len(cfg.class_names),
        "names": {i: name for i, name in enumerate(cfg.class_names)},
    }

    with open(cfg.data_yaml_path, "w") as f:
        yaml.dump(yaml_data, f)

    return cfg.data_yaml_path


# ==========================================
# STEP 4: BUILD SCALED ARCHITECTURE YAML
# ==========================================
def generate_scaled_architecture_yaml(cfg: Config = CONFIG):
    """Load default YOLOv8 base config and override its scale parameters."""
    print("\nSTEP 4: Generating custom scaled architecture YAML")

    temp_model = YOLO("yolov8n.yaml")
    arch_dict = temp_model.yaml

    # Override class count
    arch_dict["nc"] = len(cfg.class_names)

    # Ultralytics uses the 'scales' dictionary to compute module width and depth.
    arch_dict["scales"] = {
        "custom": [cfg.depth_multiple, cfg.width_multiple, 1024]
    }
    arch_dict["scale"] = "custom"

    with open(cfg.custom_model_yaml, "w") as f:
        yaml.dump(arch_dict, f)

    print(f"Generated custom architecture YAML: {cfg.custom_model_yaml}")
    print(f"Configured Channel Width Multiple: {cfg.width_multiple}")
    return cfg.custom_model_yaml


# ==========================================
# STEP 5: TRAIN MODEL FROM SCALED YAML
# ==========================================
def train_yolo_model(cfg: Config = CONFIG):
    """Train YOLO model on custom architecture YAML, or skip if weights exist."""
    print("\nSTEP 5: Training scaled YOLO model")

    if os.path.exists(cfg.best_weights_path):
        print(f"⚡ Existing best model found at: {cfg.best_weights_path}")
        print("Skipping training stage.")
        return cfg.best_weights_path

    model = YOLO(cfg.custom_model_yaml)
    model.train(
        data=cfg.data_yaml_path,
        epochs=cfg.epochs,
        patience=cfg.patience,
        imgsz=cfg.img_size,
        project=cfg.train_project,
        name=cfg.train_name,
        exist_ok=True,
    )
    print(f"Training complete. Best model saved at: {cfg.best_weights_path}")
    return cfg.best_weights_path


# ==========================================
# STEP 5b: PLOT & LOG LOSS CURVES
# ==========================================
def plot_loss_curves(cfg: Config = CONFIG):
    """Reads Ultralytics results.csv log and generates training/validation loss graphs."""
    print("\nSTEP 5b: Logging metrics and generating loss curves graph")
    results_csv_path = os.path.join(
        cfg.train_project, cfg.train_name, "results.csv"
    )

    if not os.path.exists(results_csv_path):
        print(f"⚠️ Results log CSV not found at: {results_csv_path}")
        return

    try:
        import pandas as pd
        import matplotlib.pyplot as plt

        # Read CSV and strip whitespace from column names
        df = pd.read_csv(results_csv_path)
        df.columns = [c.strip() for c in df.columns]

        epochs = df["epoch"]

        # Set up a multi-plot figure for training vs validation losses
        plt.figure(figsize=(12, 5))

        # Plot Box Loss
        plt.subplot(1, 3, 1)
        if "train/box_loss" in df.columns and "val/box_loss" in df.columns:
            plt.plot(
                epochs, df["train/box_loss"], label="Train Box Loss", color="blue"
            )
            plt.plot(
                epochs,
                df["val/box_loss"],
                label="Val Box Loss",
                color="orange",
                linestyle="--",
            )
            plt.title("Box Loss")
            plt.xlabel("Epochs")
            plt.ylabel("Loss")
            plt.legend()
            plt.grid(True)

        # Plot Class Loss
        plt.subplot(1, 3, 2)
        if "train/cls_loss" in df.columns and "val/cls_loss" in df.columns:
            plt.plot(
                epochs, df["train/cls_loss"], label="Train Cls Loss", color="green"
            )
            plt.plot(
                epochs,
                df["val/cls_loss"],
                label="Val Cls Loss",
                color="red",
                linestyle="--",
            )
            plt.title("Class Loss")
            plt.xlabel("Epochs")
            plt.ylabel("Loss")
            plt.legend()
            plt.grid(True)

        # Plot DFL Loss (Distribution Focal Loss)
        plt.subplot(1, 3, 3)
        if "train/dfl_loss" in df.columns and "val/dfl_loss" in df.columns:
            plt.plot(
                epochs, df["train/dfl_loss"], label="Train DFL Loss", color="purple"
            )
            plt.plot(
                epochs,
                df["val/dfl_loss"],
                label="Val DFL Loss",
                color="brown",
                linestyle="--",
            )
            plt.title("DFL Loss")
            plt.xlabel("Epochs")
            plt.ylabel("Loss")
            plt.legend()
            plt.grid(True)

        plt.tight_layout()
        graph_output_path = "training_validation_loss_curves.png"
        plt.savefig(graph_output_path, dpi=300, bbox_inches="tight")
        plt.close()

        print(
            f"📊 Training & Validation loss graph saved successfully as: {graph_output_path}"
        )

    except Exception as e:
        print(f"Could not generate loss curves graph: {e}")


# ==========================================
# STEP 6: MEASURE MODEL SIZE
# ==========================================
def measure_model_size(model_path=None, cfg: Config = CONFIG):
    """Measures file size of saved model weights."""
    target_path = model_path or cfg.best_weights_path
    print("\nSTEP 6: Checking Model Size")
    if os.path.exists(target_path):
        size_mb = os.path.getsize(target_path) / (1024 * 1024)
        print(f"Model File Size ({target_path}): {size_mb:.2f} MB")
        return size_mb
    else:
        print(f"Path does not exist: {target_path}")
        return None


# ==========================================
# STEP 7: EXPORT TO ONNX (WITH INT8 QUANTIZATION)
# ==========================================
def export_onnx(model_path=None, cfg: Config = CONFIG):
    """Export scaled model to ONNX format (FP32 or INT8 quantized)."""
    print(
        f"\nSTEP 7: Exporting Scaled Model to ONNX (INT8={cfg.int8_quantization})"
    )
    target_path = model_path or cfg.best_weights_path

    yolo_model = YOLO(target_path)

    exported_path = yolo_model.export(
        format="onnx",
        imgsz=cfg.img_size,
        dynamic=False,
        simplify=True,
        int8=cfg.int8_quantization,
        data=cfg.data_yaml_path,
    )

    if os.path.exists(exported_path):
        shutil.move(exported_path, cfg.onnx_output_filename)

    file_size_mb = os.path.getsize(cfg.onnx_output_filename) / (1024 * 1024)
    print(f"🎉 Quantized ONNX Model Saved: {cfg.onnx_output_filename}")
    print(f"📏 ONNX File Size: {file_size_mb:.2f} MB")
    return cfg.onnx_output_filename


# ==========================================
# MAIN EXECUTION CONTROL
# ==========================================
def main():
    # Step 1: Extract archive
    extract_dataset()

    # Step 2: Locate and Augment
    dataset_path = locate_dataset()
    process_augmentations(dataset_path)

    # Step 3: Create YOLO dataset structure & YAML
    create_yolo_dataset()

    # Step 4: Generate scaled network YAML
    generate_scaled_architecture_yaml()

    # Step 5: Train scaled YOLO Model
    best_model_path = train_yolo_model()

    # Step 5b: Log records and plot loss curves
    plot_loss_curves()

    # Step 6: Measure original size
    measure_model_size(best_model_path)

    # Step 7: Export directly to ONNX format
    export_onnx(best_model_path)


if __name__ == "__main__":
    main()

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
STEP 1: Extracting dataset zip

STEP 2a: Locating dataset

STEP 2b: Processing image augmentations

STEP 3: Creating YOLO dataset and YAML configuration

STEP 4: Generating custom scaled architecture YAML
Generated custom architecture YAML: custom_scaled_yolov8.yaml
Configured Channel Width Multiple: 0.18

STEP 5: Training scaled YOLO model
WARNING ⚠️ no model scale passed. Assuming scale='custom'.
Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, c

ONNX: export success ✅ 112.8s, saved as '/content/runs/detect/heavy_model_retrained/drowsy/weights/best_int8.onnx' (2.1 MB)

Export complete (113.1s)
Results saved to /content/runs/detect/heavy_model_retrained/drowsy/weights/best_int8.onnx
Predict:         yolo predict task=detect model=/content/runs/detect/heavy_model_retrained/drowsy/weights/best_int8.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/runs/detect/heavy_model_retrained/drowsy/weights/best_int8.onnx imgsz=640 data=yolo_augmented_dataset/data.yaml 
Visualize:       https://netron.app
🎉 Quantized ONNX Model Saved: scaled_model_int8.onnx
📏 ONNX File Size: 2.13 MB
